In [ ]:
# input
high_conf_pred="../predict_afdb/data/pred_ge_3_clique_3.tsv"
pfam_file="../_database/Pfam/Pfam-A.regions.tsv"
# output
high_conf_pfam="./tmp/entryId-pfamIds-posis.tsv"

In [2]:
import pandas as pd

df_pfam_region = pd.read_table(pfam_file, usecols=["pfamseq_acc", "pfamA_acc", "seq_start", "seq_end"])
df_pred = pd.read_table(high_conf_pred, usecols=["seq_id", "posi"])

In [3]:
id2pred_posi = dict(zip(df_pred["seq_id"].map(lambda x: x.split("-")[1]), df_pred["posi"]))
df_pfam_region = df_pfam_region[df_pfam_region["pfamseq_acc"].map(lambda x: x in id2pred_posi)]
del df_pred

In [ ]:
def get_all_positions_to_domain_id(domain_id_to_ranges_str: dict):
    result = [None for _ in range(2700)]
    for k, v in domain_id_to_ranges_str.items():
        for range_str in v.split("_"):
            start, stop = range_str.split("-")
            start, stop = int(start), int(stop)
            length = stop - start + 1
            result[(start - 1): stop] = [k for _ in range(length)]

    return result

import tqdm

records = []
for (seq_id,), df_seq in tqdm.tqdm(df_pfam_region.groupby(by=["pfamseq_acc"])):

    pfam_id2ranges = dict()
    for _, row in df_seq.iterrows():
        pfam_id = row['pfamA_acc']
        range_str = f"{row['seq_start']}-{row['seq_end']}"
        if pfam_id not in pfam_id2ranges:
            pfam_id2ranges[pfam_id] = [range_str]
        else:
            pfam_id2ranges[pfam_id].append(range_str)

    pfam_id2range_strs = dict()
    for k, v in pfam_id2ranges.items():
        pfam_id2range_strs[k] = "_".join(v)
    
    all_positions_to_domain_id = get_all_positions_to_domain_id(pfam_id2range_strs)
    pred_positions = id2pred_posi[seq_id]
    domain2positions = dict()
    for p in pred_positions.split(","):
        d = all_positions_to_domain_id[int(p)]
        if d is not None:
            if d not in domain2positions:
                domain2positions[d] = [p]
            else:
                domain2positions[d].append(p)
    
    domains = []
    positions = []
    for k, v in domain2positions.items():
        domains.append(k)
        positions.append(",".join(v))

    if len(domain2positions) > 0:
        records.append({
            "seq_id": seq_id,
            "domain": ",".join(domains),
            "posi": ";".join(positions)
        })

100%|██████████| 10420328/10420328 [36:07<00:00, 4806.87it/s] 


In [8]:
pd.DataFrame(records).to_csv(high_conf_pfam, sep="\t", index=None, header=None)